In [1]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        
        # Primary Convolutional Path F(x)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut/Skip Connection Path
        self.shortcut = nn.Sequential()
        
        # If the input and output dimensions don't match, we cannot add them directly (F(x) + x).
        # We must apply a 1x1 convolution along the shortcut path to align the channel count and stride.
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            
    def forward(self, x):
        # 1. Forward pass through the convolutional path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 2. Add the identity skip connection path directly to the output
        out += self.shortcut(x)
        
        # 3. Apply the final activation function after vector addition
        out = self.relu(out)
        return out

# Audit the structural integrity of the block
# Case 1: Standard routing (Same dimensions)
block_standard = ResidualBlock(in_channels=64, out_channels=64, stride=1)
mock_tensor = torch.randn(2, 64, 32, 32)
print(f"Standard Block Output Shape: {block_standard(mock_tensor).shape}")

# Case 2: Downsampling routing (Changing channels and spatial scale)
block_downsample = ResidualBlock(in_channels=64, out_channels=128, stride=2)
print(f"Downsampled Block Output Shape: {block_downsample(mock_tensor).shape}")

Standard Block Output Shape: torch.Size([2, 64, 32, 32])
Downsampled Block Output Shape: torch.Size([2, 128, 16, 16])
